# 🏥 Aged Care Demand Forecasting — Australian Public Sector
## Notebook 05: Modelling — XGBoost Classification + Demand Projection

> **Inputs from 04_feature_engineering.ipynb:**
> - `data/processed/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`
> - `data/processed/master_features.csv`
>
> **Note on Prophet:** AIHW data is a single snapshot (2024/25), not a quarterly time series.
> Prophet forecasting requires historical quarterly data. Part B instead uses a
> **deterministic demand projection** based on ABS population growth rates,
> which is more appropriate and defensible for policy planning with this dataset.

---

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# XGBoost + sklearn
import xgboost as xgb
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

# SHAP
import shap

import plotly.graph_objects as go

# Paths
RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
MODELS_DIR    = Path('../models')
REPORTS_DIR   = Path('../reports')
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Load processed data
X_train = pd.read_csv(PROCESSED_DIR / 'X_train.csv')
X_test  = pd.read_csv(PROCESSED_DIR / 'X_test.csv')
y_train = pd.read_csv(PROCESSED_DIR / 'y_train.csv').squeeze()
y_test  = pd.read_csv(PROCESSED_DIR / 'y_test.csv').squeeze()
master  = pd.read_csv(PROCESSED_DIR / 'master_features.csv', dtype={'sa2_code': str})

LABEL_MAP    = {0: 'Low', 1: 'Medium', 2: 'High'}
LABEL_COLORS = {'Low': '#55A868', 'Medium': '#DD8452', 'High': '#C44E52'}

print('✅ Setup complete')
print(f'   Train: {X_train.shape},  Test: {X_test.shape}')
print(f'   Master features: {master.shape}')
print(f'   Features: {list(X_train.columns)}')

---
## PART A — XGBoost Demand Risk Classifier
### A1. Train XGBoost Model

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=cv,
                             scoring='accuracy', n_jobs=-1)

print('=== XGBoost Training Results ===')
print(f'  5-Fold CV Accuracy:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'  CV Scores:           {[round(s, 4) for s in cv_scores]}')

### A2. Evaluation on Test Set

In [ ]:
y_pred  = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)

target_names = ['Low', 'Medium', 'High']

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=target_names))

auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
print(f'Macro AUC (OvR): {auc:.4f}')

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Demand Risk Classifier', fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'model_01_confusion_matrix.png', dpi=150)
plt.show()

### A3. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 8))
colors = ['#C44E52' if imp > importance_df['importance'].quantile(0.75)
          else '#4C72B0' for imp in importance_df['importance']]
ax.barh(importance_df['feature'], importance_df['importance'],
        color=colors, edgecolor='white')
ax.set_xlabel('Feature Importance (gain)')
ax.set_title('XGBoost Feature Importance — Aged Care Demand Risk', fontweight='bold')
ax.axvline(importance_df['importance'].mean(), color='gray',
           linestyle='--', alpha=0.7, label='Mean importance')
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'model_02_feature_importance.png', dpi=150)
plt.show()

### A4. SHAP Values — Model Explainability

In [ ]:
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# SHAP summary plot — High risk class
plt.figure(figsize=(9, 6))
shap.summary_plot(
    shap_values[2] if isinstance(shap_values, list) else shap_values,
    X_test,
    plot_type='bar',
    show=False,
    color='#C44E52'
)
plt.title('SHAP Feature Importance — High Risk Class', fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'model_03_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# SHAP beeswarm
plt.figure(figsize=(9, 6))
shap.summary_plot(
    shap_values[2] if isinstance(shap_values, list) else shap_values,
    X_test,
    show=False
)
plt.title('SHAP Beeswarm — High Risk Class', fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'model_04_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

### A5. SA2 Risk Map — Predicted Labels

In [ ]:
# Predict on full master dataset
X_all = master[X_train.columns]
master['predicted_risk']       = xgb_model.predict(X_all)
master['predicted_risk_label'] = master['predicted_risk'].map(LABEL_MAP)
master['prob_high']            = xgb_model.predict_proba(X_all)[:, 2]

# Summary by state
state_risk = (
    master[master['state'].notna() & (master['state'] != 'OT')]
    .groupby(['state', 'predicted_risk_label']).size()
    .unstack(fill_value=0)
)
for col in ['Low', 'Medium', 'High']:
    if col not in state_risk.columns:
        state_risk[col] = 0
state_risk = state_risk[['Low', 'Medium', 'High']]

fig, ax = plt.subplots(figsize=(10, 5))
state_risk.plot(kind='bar', ax=ax, stacked=True,
                color=['#55A868', '#DD8452', '#C44E52'],
                edgecolor='white', width=0.7)
ax.set_title('Predicted Demand Risk by State', fontweight='bold')
ax.set_ylabel('Number of SA2 regions')
ax.set_xlabel('State')
ax.legend(title='Risk Level', bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'model_05_risk_by_state.png', dpi=150)
plt.show()

# Top 10 highest risk SA2s — use actual column names from master_features
top_risk = master.nlargest(10, 'prob_high')[[
    'sa2_name', 'state', 'predicted_risk_label', 'prob_high',
    'service_gap_score', 'growth_rate_70plus_2031', 'irsd_score',
    'remoteness_cat'
]]
print('\n=== Top 10 Highest Risk SA2 Regions ===')
print(top_risk.to_string(index=False))

### A6. Save XGBoost Model

In [ ]:
xgb_model.save_model(MODELS_DIR / 'xgb_demand_risk.json')
joblib.dump(explainer, MODELS_DIR / 'shap_explainer.pkl')
master.to_csv(PROCESSED_DIR / 'master_with_predictions.csv', index=False)

print('✅ XGBoost model saved')
print(f'   Model:     models/xgb_demand_risk.json')
print(f'   Explainer: models/shap_explainer.pkl')
print(f'   Predictions added to: data/processed/master_with_predictions.csv')

---
## PART B — Deterministic Demand Projection (2024→2031→2041)

> **Why not Prophet?**
> Prophet requires a historical quarterly time series (minimum ~2 years of observations).
> The AIHW GEN data is a single annual snapshot — not a time series.
> A deterministic projection using ABS Series B population growth rates is more
> appropriate and fully defensible for government policy planning.
>
> **Approach:** Project recipients using state-level ABS growth factors already
> applied in notebook 02, combined with current SA2-level utilisation rates.

In [ ]:
# Load population projections (has pop_70plus_2031, pop_70plus_2041, growth rates)
pop_proj = pd.read_csv(RAW_DIR / 'abs_population_projections.csv', dtype={'sa2_code': str})
pop_proj['sa2_code'] = pop_proj['sa2_code'].str.zfill(9)

# Merge projected populations into master_with_predictions
proj = master.merge(
    pop_proj[['sa2_code', 'pop_70plus_2031', 'pop_70plus_2041']],
    on='sa2_code', how='left',
    suffixes=('', '_proj')
)

# Resolve column conflicts — use _proj version if original missing
for col in ['pop_70plus_2031', 'pop_70plus_2041']:
    if col + '_proj' in proj.columns:
        proj[col] = proj[col].combine_first(proj[col + '_proj'])
        proj = proj.drop(columns=[col + '_proj'])

# Current utilisation rate per SA2
proj['util_rate'] = (
    proj['total_recipients'] /
    proj['pop_70plus'].replace(0, np.nan)
).fillna(proj['total_recipients'].sum() / proj['pop_70plus'].sum())

# Projected recipients using current utilisation rate × projected population
proj['proj_recipients_2031'] = (
    proj['pop_70plus_2031'] * proj['util_rate']
).round(0).astype(int)

proj['proj_recipients_2041'] = (
    proj['pop_70plus_2041'] * proj['util_rate']
).round(0).astype(int)

proj['proj_growth_2031'] = proj['proj_recipients_2031'] - proj['total_recipients']
proj['proj_growth_2041'] = proj['proj_recipients_2041'] - proj['total_recipients']

print('=== National Demand Projection ===')
print(f'  2024 (current):  {proj["total_recipients"].sum():>10,}')
print(f'  2031 (projected):{proj["proj_recipients_2031"].sum():>10,}')
print(f'  2041 (projected):{proj["proj_recipients_2041"].sum():>10,}')
print(f'  Growth 2024→2031: +{(proj["proj_recipients_2031"].sum() / proj["total_recipients"].sum() - 1)*100:.1f}%')
print(f'  Growth 2024→2041: +{(proj["proj_recipients_2041"].sum() / proj["total_recipients"].sum() - 1)*100:.1f}%')

### B2. National Projection Chart

In [ ]:
years   = ['2024', '2031', '2041']
totals  = [
    proj['total_recipients'].sum(),
    proj['proj_recipients_2031'].sum(),
    proj['proj_recipients_2041'].sum(),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart: national totals
colors_bar = ['#4C72B0', '#DD8452', '#C44E52']
bars = axes[0].bar(years, [v / 1e6 for v in totals],
                   color=colors_bar, width=0.5, edgecolor='white')
for bar, val in zip(bars, totals):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.01,
                 f'{val / 1e6:.2f}M', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Projected Aged Care Recipients — National', fontweight='bold')
axes[0].set_ylabel('Total Recipients (millions)')
axes[0].set_ylim(0, max(totals) / 1e6 * 1.25)

# Projection by state
state_proj = (
    proj[proj['state'].notna() & (proj['state'] != 'OT')]
    .groupby('state')[['total_recipients', 'proj_recipients_2031', 'proj_recipients_2041']]
    .sum()
)
state_proj['growth_2031'] = (
    state_proj['proj_recipients_2031'] / state_proj['total_recipients'] - 1
) * 100
state_proj = state_proj.sort_values('growth_2031', ascending=True)

nat_avg = state_proj['growth_2031'].mean()
c = ['#C44E52' if g > nat_avg else '#4C72B0' for g in state_proj['growth_2031']]
axes[1].barh(state_proj.index, state_proj['growth_2031'], color=c, edgecolor='white')
axes[1].axvline(nat_avg, color='red', linestyle='--', label=f'Avg: {nat_avg:.1f}%')
axes[1].set_xlabel('Projected recipient growth 2024→2031 (%)')
axes[1].set_title('Demand Growth Rate by State', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'model_06_demand_projection.png', dpi=150)
plt.show()

### B3. Top 20 SA2 Regions by Projected Demand Growth

In [ ]:
top20_growth = (
    proj[proj['proj_growth_2031'] > 0]
    .nlargest(20, 'proj_growth_2031')
    [['sa2_name', 'state', 'predicted_risk_label', 'prob_high',
      'total_recipients', 'proj_recipients_2031', 'proj_growth_2031',
      'growth_rate_70plus_2031', 'remoteness_cat']]
    .reset_index(drop=True)
)

print('=== Top 20 SA2 Regions by Projected Demand Growth (2024→2031) ===')
print(top20_growth.to_string(index=False))

# Save projection outputs
proj_out = proj[[
    'sa2_code', 'sa2_name', 'state',
    'total_recipients', 'proj_recipients_2031', 'proj_recipients_2041',
    'proj_growth_2031', 'proj_growth_2041',
    'predicted_risk_label', 'prob_high'
]]
proj_out.to_csv(PROCESSED_DIR / 'demand_projections_sa2.csv', index=False)
print(f'\n✅ SA2 demand projections saved: {len(proj_out):,} regions')

### B4. Projection by Remoteness Category

In [ ]:
remote_labels = {1: 'Major City', 2: 'Inner Regional',
                 3: 'Outer Regional', 4: 'Remote', 5: 'Very Remote'}

remote_proj = (
    proj[proj['remoteness_cat'].notna()]
    .copy()
)
remote_proj['remoteness_label'] = remote_proj['remoteness_cat'].astype(int).map(remote_labels)

remote_summary = remote_proj.groupby('remoteness_label').agg(
    sa2_count=       ('sa2_code',             'count'),
    current=         ('total_recipients',     'sum'),
    projected_2031=  ('proj_recipients_2031', 'sum'),
    high_risk_count= ('predicted_risk_label', lambda x: (x == 'High').sum())
).reset_index()
remote_summary['growth_pct'] = (
    remote_summary['projected_2031'] / remote_summary['current'] - 1
) * 100

order = ['Major City', 'Inner Regional', 'Outer Regional', 'Remote', 'Very Remote']
remote_summary = remote_summary.set_index('remoteness_label').reindex(order).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(remote_summary['remoteness_label'],
            remote_summary['growth_pct'],
            color=['#4C72B0', '#55A868', '#DD8452', '#C44E52', '#8172B3'],
            edgecolor='white')
axes[0].set_title('Projected Demand Growth by Remoteness (2024→2031)', fontweight='bold')
axes[0].set_ylabel('Growth rate (%)')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(remote_summary['remoteness_label'],
            remote_summary['high_risk_count'],
            color='#C44E52', edgecolor='white', alpha=0.85)
axes[1].set_title('High Risk SA2 Count by Remoteness', fontweight='bold')
axes[1].set_ylabel('Number of SA2 regions')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'model_07_remoteness_projection.png', dpi=150)
plt.show()
print(remote_summary.to_string(index=False))

---
## Summary

In [ ]:
print('=== Modelling Summary ===')
print()
print('PART A — XGBoost Classifier')
print(f'  5-Fold CV Accuracy:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'  Macro AUC (OvR):     {auc:.4f}')
print(f'  High-risk SA2s:      {(master["predicted_risk_label"] == "High").sum()}')
print(f'  Medium-risk SA2s:    {(master["predicted_risk_label"] == "Medium").sum()}')
print(f'  Low-risk SA2s:       {(master["predicted_risk_label"] == "Low").sum()}')
print()
print('PART B — Deterministic Demand Projection')
print(f'  Method:              ABS Series B growth rates × SA2 utilisation rates')
print(f'  2024 recipients:     {proj["total_recipients"].sum():,}')
print(f'  2031 projected:      {proj["proj_recipients_2031"].sum():,}')
print(f'  2041 projected:      {proj["proj_recipients_2041"].sum():,}')
print(f'  Growth 2024→2031:    +{(proj["proj_recipients_2031"].sum() / proj["total_recipients"].sum() - 1)*100:.1f}%')
print()
print('Files saved:')
for f in sorted(MODELS_DIR.glob('*')) :
    print(f'  models/{f.name}')
for f in sorted(PROCESSED_DIR.glob('*.csv')):
    print(f'  processed/{f.name}')
print()
print('➡️ Next: 06_nlp_policy_analysis.ipynb')